

D? ?o?n ?? t??ng quan gi?a t? kh?a t?m ki?m v? s?n ph?m tr? v? v?i d? li?u t? HomeDepot.com
--------------------------------------------------------


??y ch? l? l? cell ???c cung c?p b?i Kaggle.  
  
N? import th? vi?n **numpy** cho vi?c t?nh to?n to?n h?c, **pandas** cho vi?c x? l? d? li?u, **matplotlib** cho vi?c v? bi?u ??, v? **re** cho vi?c x? l? k? ph?p **regex**.  
  
Ngo?i ra, kaggle c?n cho ta xem ???ng d?n ??n d? li?u v?i h?m **ls** trong linux ?? li?t k? c?c th? m?c / files trong ???ng d?n m? h? cho tr??c.

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import re

from subprocess import check_output
print(check_output(["ls", "../input"]).decode("utf8"))

**T?ng quan**
--------------
  
Home Depot ?? x?y d?ng ra b? d? li?u ???c ng??i ??nh gi? v? m?c ?? li?n quan gi?a k?t qu? t?m ki?m v? t? t?m ki?m, c? kh? n?ng thay ??i t?ch c?c ??n ch?t l??ng d?ch v?.  
  
Notebook n?y s? x?y d?ng m?t m? h?nh ??nh gi? ?? t??ng quan gi?a c?c t? kh?a t?m ki?m v? k?t qu? tr? v?, d?a tr?n d? li?u c?a Home Depot.  

# M?c l?c  
* [??c d? li?u](#read_data)  
* [Kh?m ph? d? li?u](#data_exploration)  
    - [Kh?m ph? d? li?u training](#training_data_exploration)
    - [Kh?m ph? d? li?u testing](#testing_data_exploration)
    - [Kh?m ph? d? li?u attribute](#attribute_data_exploration)
    - [Kh?m ph? d? li?u description](#description_data_exploration)
* [X? l?, ph?n t?ch, v? x?y d?ng d? li?u](#data_processing)  
    - [X?y d?ng d? li?u brand m?i, join brand v? description](#new_brand)
    - [X?y d?ng d? li?u attribute m?i](#new_attribute)
    - [T?o ra m?t s? d? li?u ??nh l??ng](#new_quantitative)
    - [Ti?n x? l?](#pre_processing)
        - [Chu?n h?a c?c x?u](#normalization)
        - [Stemming](#stem)
* [Bi?u di?n d? li?u](#data_representation)  
    - [Tf-idf](#tfidf)
    - [TruncatedSVD](#svd)
    - [Pipeline d? li?u](#tfidf_svd)
    - [H?m m?t m?t](#loss)
    - [Ph?n t?ch d? li?u train-test](#data_split)
* [Hu?n luy?n m? h?nh](#model_training)
    - [Kh?i t?o m? h?nh](#model_selection)
    - [T?m ki?m b? tham s? t?i ?u](#find_best_param)
    - [Hu?n luy?n v?i b? tham s? t?i ?u](#best_param_training)
    - [D? ?o?n k?t qu? cu?i c?ng](#prediction)
* [K?t qu? ](#result)

<a id="read_data"></a>
# ??c d? li?u  

Gi? ta s? c?n ??c d? li?u csv t? ngu?n d? li?u ???c l?u tr? d??i d?ng **.csv.zip**.  

S? d?ng h?m **pandas.read_csv**, ta s? c? kh? n?ng gi?i n?n ??ng th?i trong qu? tr?nh ??c. 

C?c file **.zip** t? ??ng ???c gi?i n?n v? ??c d??i d?ng **pd.DataFrame**.  
C?c d? li?u ???c ??c l? "**training_data**", "**testing_data**", "**attribute_data**", v? "**description**".  
  
D?a v?o t?n ta c? th? ?o?n ???c c?c d? li?u n?y l?m g?, tuy nhi?n ?? ch?c ch?n ta s? c?n kh?m ph? t?ng c?i m?t.  

In [ ]:
# Load files
training_data = pd.read_csv("../input/home-depot-product-search-relevance/train.csv.zip", encoding="ISO-8859-1")
testing_data = pd.read_csv("../input/home-depot-product-search-relevance/test.csv.zip", encoding="ISO-8859-1")
attribute_data = pd.read_csv('../input/home-depot-product-search-relevance/attributes.csv.zip')
descriptions = pd.read_csv('../input/home-depot-product-search-relevance/product_descriptions.csv.zip')

<a id="data_exploration"></a>
# Kh?m ph? d? li?u

<a id="training_data_exploration"></a>
## Kh?m ph? training_data

Hi?n ta kh?ng bi?t **training_data** mang ? ngh?a g?.  
  
Ti?p c?n ??u ti?n l? t?n c?c c?t c?a c?c d? li?u. Ta s? s? d?ng h?m **pandas.DataFrame.columns.values** tr? v? m?t list c?c t?n c?a c?c c?t trong DataFrame ??.  
  
**Training_data** c? 5 c?t, d?a v?o t?n ta c? th? suy ra ???c:  
- **product_uid**: M? c?a s?n ph?m. C?c s?n ph?m kh?c nhau s? c? m? s?n ph?m kh?c nhau.  
- **product_title**: Ti?u ?? t?n s?n ph?m.  
- **search_term**: T? kh?a t?m ki?m.  
- **relevance**: ?i?m t??ng quan nh? ?? m? t?.  
  
Nh? v?y, ??y l? d? li?u hu?n luy?n c?a ta.    

In [ ]:
print("Train data columns are: {}".format(training_data.columns.values))
print("Test data columns are: {}".format(testing_data.columns.values))
print("Attribute data columns are: {}".format(attribute_data.columns.values))
print("Description data columns are: {}".format(descriptions.columns.values))

Gi? ?? bi?t ? ngh?a c?a c?c tr??ng th?ng tin, ta mu?n hi?u r? h?n v? ki?u d? li?u trong dataframe n?y.  
  
H?m **pandas.DataFrame.info()** s? cho ta bi?t c?c c?t mang nh?ng gi? tr? nh? th? n?o.  
  
Nh?n v?o k?t qu?, ta th?y d? li?u c? **74067**, ho?n to?n kh?ng c? d? li?u **null**.  
Vi?c n?y kh? tuy?t, ta s? kh?ng ph?i x? l? d? li?u **null** trong b?ng.  
  
Tuy nhi?n, **product_title** v? **search_term** mang lo?i object, ta s? c?n xem chi ti?t h?n v? ch?ng.  

In [ ]:
training_data.info()

Ta mu?n xem m?t s? h?ng ??u c?a **product_title** v? **search_term**.  
H?m **pandas.DataFrame.head()** tr? v? 5 h?ng ??u c?a d? li?u.  
  
Ta nh?n th?y Ti?u ?? n?y ch?a c?c d? li?u v?n b?n ch?a ???c chu?n h?a, i.e ch?a c?c k? t? in hoa l?n th??ng, k? t? ??c bi?t, ... Ta s? c?n chu?n h?a l?i ?? ti?n cho vi?c x? l? sau n?y.  

In [ ]:
training_data[['product_title', 'search_term']].head()

Ta c?ng mu?n t?m hi?u d? li?u **relevance** ???c ph?n b? th? n?o.  
  
H?m **pandas.DataFrame.describe()** s? cho ch?ng ta d? li?u th?ng k? v? n?.  
  
Ph?n l?n ?? t??ng quan ??nh gi? s? n?m trong ?o?n 2.00 ??n 3.00 (50% percentile l? 2.33).  
D? li?u s? ch?nh nhau kh? l?n v? c?c m?u c? ?? t??ng quan l?n.

In [ ]:
training_data['relevance'].describe()

?? nh?n r? h?n s? ph?n b? c?a **relevance**, ta s? d?ng h?m **pandas.DataFrame.Series.hist()** ?? v? hi?u ?? histogram.  
  
K?t qu? ta nh?n th?y c?ng gi?ng nh? ?? xem ? s? li?u th?ng k?, v?i s? l??ng m?u mang ?? relevance chi?m ??i ?a s?.

In [ ]:
training_data.relevance.hist()

<a id="testing_data_exploration"></a>
L?c n?y, ta ?o?n d? li?u **testing_data** s? mang t?nh ki?m duy?t ?? n?p ra k?t qu? cu?i c?ng, do ?? n? c?ng mang d? li?u t??ng t? v?i **training_data**, ch? b? ?i tr??ng **relevance**.  
  
Ta ki?m tra b?ng h?m **pandas.DataFrame.info()** v? k?t qu? ??ng nh? v?y.  

In [ ]:
testing_data.info()

<a id="attribute_data_exploration"></a>
## Kh?m ph? d? li?u th?ng tin thu?c t?nh

Ta mu?n xem **attribute_data** mang nh?ng th?ng tin g?.  
  
Ta d?ng h?m **pandas.DataFrame.head()** ?? xem m?t s? d?ng ??u c?a d? li?u.  
  
D?a v?o k?t qu? thu ???c, ta c? m?t s? nh?n x?t sau:  
- **product_uid**: M? s?n ph?m. M? n?y l?i ???c ghi d??i d?ng s? th?p ph?n, ta s? c?n s?a l?i th?nh d?ng s? nguy?n ?? ??ng nh?t v?i c?c d? li?u kh?c. <a id="float-id"></a> 
- **name**: T?n thu?c t?nh. M?i s?n ph?m theo product_uid c? th? mang nhi?u thu?c t?nh. T?n c?c thu?c t?nh ?? ???c gi?u ?i v? thay th? b?ng c?c c?m t? g?m ch? latin in hoa, th??ng v? s?.  
- **value**: Th?ng tin v? thu?c t?nh t??ng ?ng.  Tr??ng th?ng tin n?y mang nhi?u lo?i k? t?.  
  
Nh?n v?o k?t qu?, ta s? c?n xem x?t th?m tr??ng **name**, khi th?y n? mang nh?ng gi? tr? "**bullet**" (c?c ch?m ??u d?ng thay v? m?t t? mang ? ngh?a n?o ??).

In [ ]:
print(attribute_data.head())

Nh? ?? nh?c ??n ? cell b?n tr?n, ta s? th? xem tr??ng **name** mang c?c d? li?u th? n?o.  
  
Nh?n v?i k?t qu?, ta th?y nhi?u t?n thu?c t?nh l? **Bullet**, t?c l? c?c g?ch ??u d?ng, kh?ng mang ? ngh?a g?. Tuy nhi?n, l?i c? nh?ng th?ng tin mang ng? ngh?a.  
  
C? 5411 tr??ng th?ng tin kh?c nhau, tr?n t?ng s? 2 tri?u m?u, t?c l? ta ho?n to?n c? kh? n?ng d?a v?o th?ng tin n?y ?? l?y ??c tr?ng.  

In [ ]:
print(attribute_data['name'].unique())
print(len(attribute_data['name'].unique()))

<a id="mfg"></a>
Ta s? th? xem t?n c?c thu?c t?nh xu?t hi?n v?i t?n su?t n?o.  
  
C? m?t tr??ng th?ng tin l? **MFG Brand Name** xu?t hi?n nhi?u nh?t, ng? ngh?a c?a n? l? tr??ng mang th?ng tin nh? s?n xu?t s?n ph?m ?? (**Manufacturing Brand Name**).  

??y s? l? m?t tr??ng th?ng tin r?t h?u ?ch.  

In [ ]:
attribute_counts = attribute_data.name.value_counts()
print(attribute_counts)

<a id="description_data_exploration"></a>
## Kh?m ph? d? li?u m? t?.

Hi?n ta ch?a bi?t d? li?u **description** mang th?ng tin g?.  

Ta s? in ra m?t s? d?ng ??u c?a d? li?u m? t? b?ng h?m **pandas.DataFrame.head()**.  
  
Nh?n v?o d? li?u, ta th? d? li?u n?y mang 2 th?ng tin:  
    - **product_uid**: M? s?n ph?m.  
    - **product_description**: D? li?u v?n b?n m? t? s?n ph?m. D? li?u n?y bao g?m c? c?c k? t? latin vi?t hoa, vi?t th??ng, c?c ch? s? v? k? t? ??c bi?t.    
    
D? li?u n?y kh?ng c? g? ??c bi?t l?m. Tuy nhi?n, tr??ng **product_desciption** c? th? mang nh?ng t? li?n quan ??n **search_term**, ta s? t?m hi?u sau.  

In [ ]:
descriptions.head()

<a id="data_processing"></a>
# X? l?, ph?n t?ch, v? x?y d?ng d? li?u

<a id="new_brand"></a>
## X?y d?ng th?ng tin brand, v? join d? li?u brand v? description

Tr??c ti?n, ta th?y **training_data** v? **testing_data** ??u mang th?ng tin gi?ng nhau (ch? kh?c l? **testing_data** kh?ng c? tr??ng **relevance**).  

Do ?? ?? x? l? ta s? n?i ch?ng v?i nhau ?? l?m m?t th?, h?n l? chia nhau ?? x? l?.  

H?m **pandas.concat** s? n?i 2 DataFrame v?i nhau, tr??ng th?ng tin b? thi?u s? ???c thay b?ng **NaN**.  
  
K?t qu? thu ???c l? m?t DataFrame t?ng h?p c?a **training_data** v? **testing_data**. Ta c? th? nh?n qua m?t ch?t v? ch?ng qua h?m **all_data.tail()** (**relevance** c?a **tesing_data** ???c ?p v? NaN), v? c? t?ng c?ng 240760 h?ng t?t c?.

In [ ]:
all_data = pd.concat([training_data, testing_data], axis=0, ignore_index=True)
print(all_data.tail())
len(all_data)

Nh? ?? nh?c ??n ? [??y](#mfg), **MFG Brand Name** l? m?t th?ng tin h?u ?ch.  
  
? ??y t?i mu?n x?y d?ng m?t dataFrame kh?c ch? d?a tr?n th?ng tin n?y, b?ng vi?c ch?n nh?ng h?ng ? **attribute_data** c? **attribute_data.name == "MFG Brand Name"**, l?y **product_uid** v? "**value**", v? tr??ng **value** ???c thay b?ng th?ng tin m?i: **brand** - t?n nh?n hi?u.  
  
K?t qu? thu ???c l? **brand_data**, mang th?ng tin nh?n hi?u c?a c?c s?n ph?m, ta c? th? nh?n qua ch?ng m?t ch?t qua h?m **brand_data.head()**.  

In [ ]:
brand_data = attribute_data[attribute_data.name == "MFG Brand Name"][["product_uid", "value"]].rename(columns={"value": "brand"})
brand_data.head()

D? li?u **search_term** c? ch?a s?, v? pandas c? th? ?p ki?u n? th?nh s? ? m?t s? h?ng.  
Trong khi r? r?ng ta mu?n **search_term** l? x?u, ta c?n ?p ki?u ch?ng th?nh **str**.  
  
Ta d?ng h?m **astype(str)** ?? ?p ch?ng th?nh **str**.  
  
K?t qu? thu ???c l? m?i **search_term** ??u l? x?u, thu?n ti?n cho sau n?y.  
  
Ngo?i ra, ta c?ng l?m t??ng t? v?i th?ng tin brand c?a **brand_data**, sau khi lo?i b? m?i d? li?u **na**.

In [ ]:
all_data["search_term"]= all_data["search_term"].astype(str)
brand_data.dropna(inplace=True)
brand_data['brand'] = brand_data['brand'].astype(str)

Nh? ?? n?i ? [??y](#float-id), ta s? mu?n ?p ki?u **product_uid** c?a **attribute_data** v? d?ng **int** ?? chu?n h?n trong qu? tr?nh chuy?n h?a sau n?y.  
  
Ngo?i ra, ta c?ng lo?i b? c?c tr??ng **na** ? **attribute_data** v? **descriptions** lu?n.  
  
K?t qu? l? **attribute_data** v? **descriptions** ??u ch? mang c?c d? li?u chu?n.

In [ ]:
attribute_data.dropna(inplace=True)
attribute_data['product_uid'] = attribute_data['product_uid'].astype(int)
descriptions.dropna(inplace=True)

V?n ?? ? ??y l?, **descritions** v? **brand_data** ??u l? nh?ng th?ng tin h?u ?ch, nh?ng n? l?i n?m ? c?c DataFrame kh?c nhau, kh? kh?n cho vi?c s? d?ng.  
  
??u s? h?u tr??ng **product_uid**, ta ho?n to?n c? th? join ch?ng v?i **all_data**, b?ng **left_join**.   
  
K?t qu? l? **all_data**, x?t theo **product_uid**, **all_data** s? ???c "g?n" th?m th?ng tin **descritions** v? **brand_data** t??ng ?ng. Ta c? th? d? d?ng xem ch?ng v?i h?m **all_data.head()**.    

In [ ]:
all_data = pd.merge(all_data, descriptions, how='left', on='product_uid')
all_data = pd.merge(all_data, brand_data, how='left', on='product_uid')
all_data.head()

<a id="new_attribute"></a>
### X?y d?ng attribute_data m?i

Hi?n ta mu?n x?y d?ng m?t tr??ng **attribute_data** m?i, ch? g?m 2 tr??ng l? **product_uid** v? **attribute** (text).  
  
Nh? ?? n?i ??n ? [??y](#mfg), th?ng tin "**Bullet**" l? v? ngh?a v? t?i mu?n x?a ch?ng.  
Tr??c ti?n, t?i t?o **attribute_data_stripped** m?i (?? tr?nh thay ??i d? li?u g?c), v?i h?m **re.sub** ?? x?a ?i c?c "**Bullet**", thay b?ng x?u r?ng.

In [ ]:
attribute_data_stripped = attribute_data
attribute_data_stripped['name'] = attribute_data_stripped['name'].apply(lambda s: re.sub(r"Bullet([0-9]+)", "", s))

Sau b??c b?n tr?n, gi? attribute s? c?n mang d? li?u t? c? **name** v? **value**, do ?? ta c?ng x?u l?i.  
  
Ta x?y d?ng tr??ng **attribute** b?ng vi?c c?ng x?u, v?i kho?ng tr?ng ? gi?a, gi?a t?n **attribute** v? tr??ng **value**.  
  
Sau b??c n?y, d? li?u **attribute** ?? c? v? s?n s?ng ?? t?i DataFrame m?i.  

In [ ]:
attribute_data_stripped['attribute'] = attribute_data_stripped['name'] + " " + attribute_data_stripped['value']

Sau b??c tr?n, ta v?n th?y c? v?n ??, ?? l? m?t **product_uid** ch?a nhi?u **attribute**.  
  
Do ??, ta nh?m h?t c?c d? li?u b?ng h?m **groupby('product_uid')**, sau ?? c?ng x?u ch?ng l?i b?ng h?m **' '.join(...)** v?i **agg()**, ph?n t?ch b?i d?u c?ch.  
  
K?t qu? l? **attribute_data_new**, v?i 2 tr??ng l? **product_uid** v? **attribute**, s?n s?ng cho vi?c g?p v?o d? li?u ch?nh **all_data** nh? **brand_data** v? **descriptions**.

In [ ]:
attribute_data_new = attribute_data_stripped.groupby('product_uid').agg({'attribute': lambda s : ' '.join(s.astype(str))}).reset_index()
attribute_data_new.head()

<a id="normalize"></a>
**attribute_data** v?n ??ng ??c l?p, gi? ta c?n g?p n? v?o v?i **all_data**.  
  
Gi?ng nh? v?i brand_data v? descriptions, ta s? th?c hi?n **left_join** gi?a **all_data** v? **attribue_data_new**, d?a tr?n **product_uid** b?ng h?m **pandas.merge()**.  
  
K?t qu? thu ???c l? **all_data** v?i th?m th?ng tin **attribute**.

In [ ]:
all_data = pd.merge(all_data, attribute_data_new, how = 'left', on = 'product_uid')
all_data.head()

<a id="pre_processing"></a>
## Ti?n x? l?

Nh?c ??n vi?c ti?n x? l?, ta ngh? ??n m?t s? b??c ph? bi?n sau:  
* Chu?n h?a x?u (vi?t th??ng h?t, lo?i b? c?c k? t? ??c bi?t, ....)  
* Thay nh?ng t? c?ng ngh?a nh?ng c? c?ch vi?t kh?c (v? d? nh? s? nhi?u, s? ?t, chia th?, ...)  
  
Tr??c ti?n ta s? Import nh?ng th? vi?n c?n thi?t, chu?n b? c?c h?m cho vi?c x? l?.  

In [ ]:
from nltk.stem.snowball import SnowballStemmer
import re

<a id="normalization"></a>
### Chu?n h?a x?u

Nh? ?? nh?c ??n b?n tr?n, ??u ti?n l? vi?c chu?n h?a x?u.  
  
Ta x?y d?ng h?ng **normalize_string** v?i input l? 1 x?u, k?t qu? tr? v? m?t x?u ?? chu?n h?a.  
C?c b??c chu?n h?a ???c ch? th?ch t?ng d?ng trong code.  
?? vi?c x? l? thu?n ti?n, ta s? d?ng k? ph?p **regex** v? th? vi?n **re** c?a python.  
Vi?c n?y gi?p t?c ?? x? l? x?u ??t cao nh?t.  
  
Sau cell n?y ta ?? c? h?m chu?n h?a x?u **normalize_string**, sau n?y n?u mu?n d?ng ta ch? vi?c **apply** n? cho c?c thu?c t?nh c?a DataFrame.

In [ ]:
def normalize_string(input_str: str):
    # Lo?i b? t?t c? c?c k? t? ??c bi?t
    input_str = re.sub('\W', ' ', input_str.lower())
    # Lo?i b? c?c k? t? ??ng m?t m?nh
    input_str = re.sub('\s+[a-zA-Z]\s+', ' ', input_str)
    # Sau b??c tr?n, c?n s?t l?i c?c k? t? ??ng 1 m?nh ? ??u c?u
    # Ta x?a n?t tr??ng h?p n?y
    input_str = re.sub(r'\^[a-zA-Z]\s+', ' ', input_str) 
    # S? c? nhi?u d?u c?ch ??ng c?nh nhau sau b??c tr?n 
    # Ta s? thay to?n b? c?c d?u c?ch ??ng c?nh nhau th?nh 1 d?u duy nh?t 
    input_str = re.sub('\s+', ' ', input_str, flags=re.I)
    
    # C?c s? c?ng r?t quan tr?ng, tuy nhi?n ta kh?ng mu?n c?c s? xu?t hi?n trong x?u.
    # Ta thay c?c ch? s? th?nh c?c ch? ti?ng Anh t??ng ?ng
    input_str = re.sub(r"zero\.?", r"0 ", input_str)
    input_str = re.sub(r"one\.?", r"1 ", input_str)
    input_str = re.sub(r"two\.?", r"2 ", input_str)
    input_str = re.sub(r"three\.?", r"3 ", input_str)
    input_str = re.sub(r"four\.?", r"4 ", input_str)
    input_str = re.sub(r"five\.?", r"5 ", input_str)
    input_str = re.sub(r"six\.?", r"6 ", input_str)
    input_str = re.sub(r"seven\.?", r"7 ", input_str)
    input_str = re.sub(r"eight\.?", r"8 ", input_str)
    input_str = re.sub(r"nine\.?", r"9 ", input_str)

    return input_str

T? [??y](#normalize), ta s? chu?n h?a m?i th?ng tin l? x?u, bao g?m:  
* **search_term**  
* **product_title**  
* **product_description**  
* **attribute**  
?i?u n?y c? th? ???c th?c hi?n b?ng h?m **apply(normalize_string)**.  
  
Do th?i gian ch?y kh? l?u n?n ta import th?m th? vi?n time ?? t?nh xem ?? m?t bao nhi?u th?i gian.  
  
K?t qu? thu ???c l? **all_data** ?? ???c chu?n h?a m?i x?u.

In [ ]:
import time

start_time = time.time()

all_data['search_term'] = all_data['search_term'].apply(str).apply(normalize_string)
all_data['product_title'] = all_data['product_title'].apply(str).apply(normalize_string)
all_data['product_description'] = all_data['product_description'].apply(str).apply(normalize_string)
all_data['attribute'] = all_data['attribute'].apply(str).apply(normalize_string)
print(all_data.head())

print("--- %s seconds ---" % (time.time() - start_time))

<a id="stem"></a>
### Stemming

Nh? ?? nh?c ??n, ta mu?n thay nh?ng t? g?n gi?ng nhau ch? b?ng 1 t? duy nh?t (v? d? nh? s? nhi?u, s? ?t, chia th?, ...), ??y g?i l? qu? tr?nh "**stemmed**".  
  
Ta s? s? d?ng th? vi?n ?? h? tr? c?a **nltk**, ?? l? **SnowballStemmer** v?i ng?n ng? l? english.  
V?i m?i t? c? ???c b?ng h?m **str.split()**, ta thay n? b?ng t? g?c (n?u c?).  
  
K?t qu? ta ???c h?m **str_stemmer**, nh?n ??u v?o l? m?t x?u, v? tr? v? m?t x?u ?? ???c "**stemmed**".  

In [ ]:
stemmer = SnowballStemmer('english')

def str_stemmer(s):
    return " ".join([stemmer.stem(word) for word in s.lower().split()])

Gi? ta ?p d?ng h?m tr?n v?o **all_data** v?i h?m **apply(str_stemmer)**.  
  
T??ng t? nh? ph?n chu?n h?a x?u, ta c?ng ?o th?i gian ch?y.  
  
K?t qu? thu ???c l? **all_data** v?i c?c x?u ?? ch? ch?a nh?ng t? g?c, ch? kh?ng c?n bi?n th? n?a.

In [ ]:
start_time = time.time()

all_data['search_term'] = all_data['search_term'].apply(str).apply(str_stemmer)
all_data['product_title'] = all_data['product_title'].apply(str).apply(str_stemmer)
all_data['product_description'] = all_data['product_description'].apply(str).apply(str_stemmer)
all_data['attribute'] = all_data['attribute'].apply(str).apply(str_stemmer)

print("--- %s seconds ---" % (time.time() - start_time))

<a id="new_quantitative"></a>
## T?o ra m?t s? d? li?u ??nh l??ng

?? c? th? x?y d?ng th?m c?c tr??ng th?ng tin, ta c?n bi?t n? c? ?nh h??ng th? n?o ??n **relevance** - y?u t? ta c?n ?o?n.  
  
?? ? ??n ?? d?i c?a c?c tr??ng th?ng tin v?n b?n, ta s? v? th? ch?ng tr?n ?? th? v? 1 ???ng **regression**.  
Ta s? d?ng th? vi?n **matplotlib** ?? v? ?? th? n?y.  
Chi ti?t h?n ???c gi?i th?ch trong code.
  
Ta ???c m?t h?m see_correlation v? ra s? t??ng quan gi?a ?? d?i c?a m?t tr??ng th?ng tin b?t k? v?i **relevance**. Tham s? **transform** s? b?ng **True** n?u ta mu?n l?y ?? d?i c?a **data_field**, v? b?ng **False** n?u ta mu?n l?y lu?n gi? tr? c?a **data_field**.

In [ ]:
import matplotlib.pyplot as plt

def see_correlation(data_field, transform=True):
    # Ta ch? l?y 1 ph?n t? d? li?u ban ??u ?? cho hi?u qu? t?t nh?t
    # V? ta ch? l?y ph?n c?a training_data, th? ch?a relevance
    data_sample = all_data[:len(training_data) - 1].sample(frac=.1)
    # L?y ?? d?i c?a tr??ng th?ng tin
    x_ar = np.array(data_sample[data_field].map(lambda x:len(str(x).split())).astype(np.int64)) if transform else data_sample[data_field]
    # L?y relevance t??ng ?ng
    y_ar = np.array(data_sample['relevance'])
    # V? c?c ?i?m d? li?u l?n ?? th?
    plt.plot(x_ar, y_ar, 'o')
    # T?m tham s? m v? b ?? v? ???ng regression
    m, b = np.polyfit(x_ar, y_ar, 1)
    # V? ???c regression
    plt.plot(x_ar, m * x_ar + b)

Nh? ?? gi?i th?ch ? tr?n, gi? ta s? v? th? ra s? t??ng quan c?a **relevance** v?i l?n l??t ?? d?i c?a t?ng tr??ng th?ng tin, b?t ??u v?i **product_title**.  
  
Theo k?t qu?, c?c ?i?m **relevance** ph?n b? r?t ??u nhau tr?n m?i ?? d?i **product_title**, ???ng regression g?n nh? ?i ngang. Ta c? th? nh?n x?t r?ng **product_title** g?n nh? kh?ng ?nh h??ng g? ??n **relevance** c?.  
  
Do ??, do s? kh?ng kh? quan, sau n?y ta s? kh?ng s? d?ng ??c tr?ng n?y v?o kh?u hu?n luy?n.

In [ ]:
see_correlation('product_title')

**Product_title** kh?ng mang l?i k?t qu?, ta ti?p t?c v?i **search_term**.  
  
Theo k?t qu?, c?c ?i?m relevance ph?n b? kh? ??u tr?n m?i ?? d?i **search_term**, tuy nhi?n l?i c? nhi?u ?i?m kh?c th??ng m? theo xu h??ng n?o ??. ???ng regression c? ?? d?c kh? l?n.  
  
Ta c? th? nh?n x?t r?ng **search_term** c? ?nh h??ng ??n **relevance**.  
  
Do ??, v?i k?t qu? kh? quan kh? quan, ta s? s? d?ng ??c tr?ng n?y v?o kh?u hu?n luy?n.  

In [ ]:
see_correlation('search_term')

Ta ti?p t?c v?i **product_description**.  
  
Theo k?t qu?, c?c ?i?m relevance ph?n b? kh? ??u tr?n m?i ?? d?i **product_description**, tuy nhi?n l?i c? nhi?u ?i?m kh?c th??ng m? theo xu h??ng n?o ??.  
???ng regression c? ?? d?c ??ng c?ng kh? ??ng k?.  
  
Ta c? th? nh?n x?t r?ng **product_description** c? ?nh h??ng ??n **relevance**.  
  
Do ??, c?ng v?i k?t qu? kh? quan kh? quan, ta s? s? d?ng ??c tr?ng n?y v?o kh?u hu?n luy?n. 

In [ ]:
see_correlation('product_description')

Ta ti?p t?c v?i **brand**.  
  
Theo k?t qu?, c?c ?i?m **relevance** ph?n b? kh? ??u tr?n m?i ?? d?i **brand**, tuy nhi?n l?i c? nhi?u ?i?m kh?c th??ng m? theo xu h??ng n?o ??.  
???ng regression c? ?? d?c ??ng c?ng kh? ??ng k?.  
  
Ta c? th? nh?n x?t r?ng brand c? ?nh h??ng ??n **relevance**.  
  
Do ??, c?ng v?i k?t qu? kh? quan kh? quan, ta s? s? d?ng ??c tr?ng n?y v?o kh?u hu?n luy?n. 

In [ ]:
see_correlation('brand')

Ta ti?p t?c v?i **attribute**.  
  
Theo k?t qu?, c?c ?i?m relevance ph?n b? kh? ??u tr?n m?i ?? d?i **attribute**, tuy nhi?n l?i c? nhi?u ?i?m kh?c th??ng m? theo xu h??ng n?o ??.  
???ng regression c? ?? d?c ??ng c?ng kh? ??ng k?.  
  
Ta c? th? nh?n x?t r?ng **attribute** c? ?nh h??ng ??n **relevance**.  
  
Do ??, c?ng v?i k?t qu? kh? quan kh? quan, ta s? s? d?ng ??c tr?ng n?y v?o kh?u hu?n luy?n. 

In [ ]:
see_correlation('attribute')

D?a v?o c?c ?? th? regression tr?n, ta th?y chi?u d?i c?a c?c th?ng tin, tr? **title** v? **attribute** c?ng c? ?nh h??ng ??n relevance.  
  
Do ?? ta ??a th?ng tin chi?u d?i title b?ng h?m **len(title)** v?o th?nh m?t tr??ng th?ng tin kh?c.  
?? d?i c?a **query**, **brand**, **descrition**, **attribute** c?ng t??ng t?.  
  
Ta th?m v?o **all_data** c?c th?ng tin sau, l? ?? d?i c?c th?ng tin n?i sinh c?a d? li?u, bao g?m:
* **len_of_query**: S? t? (ng?n c?ch b?i d?u c?ch) trong c?m t? t?m ki?m  
* **len_of_brand**: ?? d?i s? t? trong t?n nh?n hi?u  
* **len_of_description**: ?? d?i ?o?n v?n b?n m? t? c?a s?n ph?m  

In [ ]:
all_data['len_of_query'] = all_data['search_term'].map(lambda x:len(str(x).split())).astype(np.int64)

all_data['len_of_brand'] = all_data['brand'].map(lambda x:len(str(x).split())).astype(np.int64)

all_data['len_of_description'] = all_data['product_description'].map(lambda x:len(str(x).split())).astype(np.int64)

D?a v?o ch?nh th?c t? s? d?ng c?c c?ng c? t?m ki?m, ta ?o?n r?ng c?c search engine th??ng d?a v?o c?c t? trong **search_term** m? xu?t hi?n trong v?n b?n.  
  
Do ??, ta s? ??m s? l??ng c?c t? n?m trong t? kh?a t?m ki?m (**search_term**) m? xu?t hi?n trong:  
* Ti?u ??: **word_in_title**  
* M? t? s?n ph?m: **word_in_description**  
  
?? l?m ???c ?i?u n?y, ta ??nh ngh?a h?m **str_common_word**, nh?n v?o 2 x?u, tr? v? s? l??ng t? trong x?u 1 ?? xu?t hi?n trong x?u 2.  

In [ ]:
def str_common_word(str1, str2):
    return sum([1 if str2.find(word) >= 0 else 0 for word in set(str1.split())])

Tr??c ti?n ta t?o m?t tr??ng th?ng tin l? **product_info** ?? ti?n x? l? sau n?y.  
Th?ng tin n?y ???c g?p t? t?t c? c?c d? li?u mang th?ng tin s?n ph?m:  
* **search_term**: C?m t? t?m ki?m.  
* **product_title**: Ti?u ?? s?n ph?m  
* **product_description**: M? t? s?n ph?m  
  
Ta g?p ch?ng l?i, ph?n t?ch b?ng k? t? "\t", ?? sau n?y ti?n t?ch ra.  
  
K?t qu? thu ???c l? **all_data** v?i tr??ng **product_info**.

In [ ]:
all_data['product_info'] = all_data['search_term'] + '\t' + all_data['product_title'] + '\t' + all_data['product_description']

Gi? ta c?n t?nh **word_in_title** v? **word_in_description** m? ?? nh?c ??n b?n tr?n.  
  
T? **product_info**, ??u ti?n ta t?ch 3 th?ng tin ra b?ng h?m split('\t'), th? k?t qu? ??u ti?n l? **search_term**, th? 2 l? **title**, th? 3 l? **description**.  
Ti?p ??, ta s? d?ng h?m **str_common_word** ?? ??nh ngh?a ?? ??m s? t? trong **search_term** trong l?n l??t 2 th?ng tin **title** v? **description**.  
  
K?t qu? l? 2 tr??ng th?ng tin m?i trong **all_data**, l? **word_in_title** v? **word_in_description**.

In [ ]:
all_data['word_in_title'] = all_data['product_info'].map(lambda x:str_common_word(str(x).split('\t')[0], str(x).split('\t')[1]))
all_data['word_in_description'] = all_data['product_info'].map(lambda x:str_common_word(str(x).split('\t')[0], str(x).split('\t')[2]))

Gi? ta mu?n ??nh gi? xem c?c th?ng tin m?i n?y c? ?nh h??ng ??n **relevance** kh?ng.
  
Ta d?ng l?i h?m **see_correlation**, v?i tham s? th? 2 l? **False**, t?c l? kh?ng c?n l?y **len** - ?? d?i v?n b?n n?a m? l?y tr?c ti?p d? li?u lu?n.  
  
B?t ??u v?i **word_in_description**, v?i ???ng regression c? ?? d?c r?t l?n, ta th?y n? ?nh h??ng kh? nhi?u ??n **relevance**.  
  
??y s? l? m?t th?ng tin h?u ?ch m? ta s? d?ng cho vi?c hu?n luy?n.

In [ ]:
see_correlation('word_in_description', False)

Ti?p t?c v?i **word_in_title**, l?m t??ng t? b?n tr?n.  
  
Ta th?y n? ?nh h??ng kh? l?n ??n relevance khi ???ng regression c? ?? d?c kh? l?n, ??y c?ng s? l? m?t th?ng tin h?u ?ch.

In [ ]:
see_correlation('word_in_title', False)

Ta th?y m?t v?n ??, ?? l? s? l??ng c? th? kh?ng ph?i l? m?t con s? c? th? v? s? t??ng quan gi?a c?c d? li?u.  
Ta c?n c?n ph?i x?t ??n t? l? xu?t hi?n trong c?c ng? c?nh n?a.  
C? th?, d?a v?o s? ??m v?a t?nh b?n tr?n, ta t?nh t? l? c?c t? xu?t hi?n trong **search_term** c?a c?c tr??ng th?ng tin sau:  
* Ti?u ??: **ratio_title**  
* M? t?: **ratio_description**  
Vi?c t?nh 2 ch? s? n?y c?ng ??n gi?n, ta ch? c?n l?y th??ng c?a 2 tr??ng th?ng tin ?? c?, **word_in_title** / **word_in_description** t??ng ?ng, v?i m?u l? **all_of_query**.
  
K?t qu? l? **all_data** v?i th?m 2 tr??ng, l? **ratio_title** v? **ratio_description**.

In [ ]:
all_data['ratio_title'] = all_data['word_in_title'] / all_data['len_of_query']
all_data['ratio_description'] = all_data['word_in_description'] / all_data['len_of_query']

Gi? ta mu?n bi?t ch?ng c? ?nh h??ng ??n **relevance** hay kh?ng. Ta b?t ??u v?i **ratio_title**.  
  
Ta ti?p t?c s? d?ng h?m **see_correlation** nh? tr?n.  
  
???ng regression ch?y t? **relevance** b?ng **2.0** ??n g?n **2.75**, k?t qu? cho th?y ??y l? m?t th?ng tin r?t h?u ?ch, v? ta c? th? s? d?ng th?ng tin n?y nh? m?t feature cho vi?c hu?n luy?n.

In [ ]:
see_correlation('ratio_title', False)

Gi? ta mu?n bi?t **ratio_description** c? ?nh h??ng ??n **relevance** hay kh?ng.  
  
Ta ti?p t?c s? d?ng h?m **see_correlation** nh? tr?n.  
  
???ng regression ch?y t? **relevance** b?ng **2.0** ??n h?n **2.5**, k?t qu? cho th?y ??y l? m?t th?ng tin r?t h?u ?ch, v? ta c? th? s? d?ng th?ng tin n?y nh? m?t feature cho vi?c hu?n luy?n.

In [ ]:
see_correlation('ratio_description', False)

Hi?n t?i, c?n duy nh?t m?t th?ng tin m? ta ch?a nh?c ??n, ?? l? b?n th?n v?n b?n c?a **description**, **brand**, v? **attribute**.  
  
?? s? d?ng cho b??c sau, ta g?p ti?p 3 tr??ng th?ng tin: **descriptions**, **brand**, **attribute** l?i th?nh m?t.  
  
K?t qu? l? **all_data** c? th?m tr??ng **prod_desc_merge**.

In [ ]:
all_data["prod_desc_merge"] = all_data["product_description"].map(str) +' ' + all_data["brand"].fillna('') + ' ' + all_data["attribute"].fillna('').map(str)

Ta nh?n th?y trong qu? tr?nh, c? r?t nhi?u c?t ?? ???c sinh m?i ra nh?ng kh?ng d?ng ??n, ho?c ch?ng ?? ???c g?p l?i ?? ti?n x? l?.  
  
Ta t?o m?t cells ?? x?a ch?ng ?i, s? d?ng l?m **pandas.DataFrame.drop(inplace=True, axis=1)**, t?c l? x?a theo c?t, v? x?a lu?n trong **all_data**.  
  
K?t qu? thu ???c l? **all_data** ch? c?n c?c tr??ng th?ng tin m? ta cho l? h?u ?ch.

In [ ]:
for column in ["attr", "product_description", "brand", "attribute", "product_attributes", "product_info", 'last_word_title_match', 'first_word_title_match', 'last_word_description_match', 'first_word_description_match']:
    if column in all_data:
        if column in all_data:
            all_data.drop(column, axis=1, inplace=True)

<a id="data_representation"></a>
# Bi?u di?n d? li?u

<a id="tfidf"></a>
## TF-IDF

Hi?n ta c?n bi?u di?n c?c d? li?u x?u cu?i c?ng th?nh s?, bao g?m **search_term**, **product_title**, v? **prod_desc_merge**.  
  
? b??c n?y, ta s? d?ng **tf-idf**.  
  
**TF-iDF** l? m?t ph?p th?ng k?, v?i t?c d?ng ph?n ?nh t?m quan tr?ng c?a m?t t? / d?u hi?u (token) v?i t?p h?p v?n b?n. Gi? tr? c?a **tf-idf** t?ng t? l? thu?n v?i s? l?n xu?t hi?n c?a t? ?? trong t?i li?u, v? ???c gi?m ?i v?i s? l??ng c?c v?n b?n ch?a t? / d?u hi?u n?y. M? h?nh n?y c?ng ???c cung c?p trong th? vi?n scikit-learn.  
  
K?t qu? thu ???c sau cell n?y l? m?t object **tfidf** v?i **ngram_range** l? 1 ??n 2, t?c l? x?t c?c t? ??n v? t? gh?p 2 t?, c?ng v?i ?? l? lo?i b? c?c **stop_words** trong ti?ng Anh.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')

<a id="svd"></a>
## TruncatedSVD

S? chi?u c?a c?c d? li?u c?ng r?t l?n.  
  
?? x? l? vi?c n?y, l?m cho t?c ?? x? l? ???c nhanh h?n, ta s? d?ng **TruncatedSVD** ?? gi?m chi?u c?a d? li?u.  
  
K?t qu? sau cell n?y l? m?t object c?a **truncatedSVD** v?i s? chi?u l? 100 - con s? t?i th? nghi?m m? ?em l?i k?t qu? kh? quan trong th?i gian ch?p nh?n ???c.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state = 2021)

<a id="tfidf_svd"></a>
## Pipeline c?a d? li?u

Gi? ta mu?n d? li?u s? ???c x? l? l?n l??t b?ng **tfidf** v? **svd**, tuy nhi?n vi?c vi?t fit 2 l?n s? l?m ta code b?t ti?n h?n.  
  
Do ?? ta s? d?ng **PipeLine** t? th? vi?n **sklearn**, cho ta n?i **tfidf** v?i **svd**.  
  
K?t qu? c?a cell l? m?t pipeline, cho ph?p d? li?u ??u v?o ?i qua m?i kh?u trong **pipeline**, r?t thu?n ti?n cho vi?c x? l?.

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[('tfidf', tfidf), ('svd', svd)])

? b??c n?y, nh? ?? n?i, ta x? l?:
* **product_title**  
* **search_term**  
* **prod_desc_merge**  
v?i pipeline ? tr?n, th?ng qua h?m **pipe.fit_transform(input_data)**.  
  
K?t qu? l? c?c tr??ng th?ng tin tr?n ???c bi?u di?n d??i d?ng m?y c? th? d? d?ng hi?u ???c, chu?n b? cho kh?u hu?n luy?n.  
  
Ngo?i ra, kh?u n?y c?ng s? m?t nhi?u th?i gian n?n t?i mu?n ?o th?i gian ch?y b?ng th? vi?n time.

In [ ]:
start_time = time.time()

all_data["product_title"] = pipe.fit_transform(all_data["product_title"])
all_data["search_term"] = pipe.fit_transform(all_data["search_term"])
all_data["prod_desc_merge"] = pipe.fit_transform(all_data["prod_desc_merge"])

print("--- %s seconds ---" % (time.time() - start_time))

In [ ]:
# import pickle

# with open('processed_transformed.data', 'wb') as picklefile:
#     pickle.dump(all_data, picklefile)

In [ ]:
# import pickle

# with open('../input/processed-data-sets/processed_transformed.data', 'rb') as object_to_be_loaded:
#     all_data = pickle.load(object_to_be_loaded)
    
# print(len(all_data))
# all_data.head()

<a id="loss"></a>
## H?m m?t m?t

Gi? ta c?n m?t h?m ??nh gi? sai s? c?a m? h?nh.  
  
Nh? ?? b?i ?? m? t? c?ch ??nh gi? k?t qu?, ta c?ng s? d?ng h?m **mean_squared_error**, ?? ???c cung c?p trong th? vi?n scikit-learn.  
Ta c?ng s? d?ng h?m **make_scorer** trong th? vi?n n?y ?? t?o ra m?t ??i t??ng ?? th? vi?n hu?n luy?n g?i ???c, v?i tham s? **greater_is_better=False**, t?c l? h?m c?ng tr? v? k?t qu? nh? th? c?ng t?t.  
  
K?t qu? c?a cell n?y l? ??nh ngh?a h?m **fmean_square_error** v? ??i t??ng **RMSE** ?? sau n?y s? d?ng cho vi?c ??nh gi? sai s? c?a m? h?nh.

In [ ]:
from sklearn.metrics import mean_squared_error, make_scorer

def fmean_squared_error(ground_truth, predictions):
    fmean_squared_error_ = mean_squared_error(ground_truth, predictions)**0.1
    return fmean_squared_error_

RMSE = make_scorer(fmean_squared_error, greater_is_better=False)

<a id="data_split"></a>
## Ph?n t?ch d? li?u train-test

Hi?n gi? m?i d? li?u ??u ?? ???c x? l?, tuy nhi?n ch?ng ta l?i c?n t?ch ra d??i d?ng **training_data** v? **testing_data**.  
  
Ta d?ng h?m **all_data.iloc** ?? t?ch l?i d? li?u **train** v? **test**.  
B?i th? t? c?a c?c m?u kh?ng thay ??i, ta ch? ??n gi?n l? l?y theo s? l??ng m?u trong **training_data**.  
  
K?t qu? sau cell n?y l? 2 dataFrames, **df_train** - d? li?u d?ng ?? train, v? **df_test** - d? li?u test ?? l?y k?t qu? n?p b?i.

In [ ]:
df_train = all_data.iloc[:len(training_data)]
df_test = all_data.iloc[len(training_data):]

Gi? ta c?n t?ch r? r?ng **X_train**, **X_test**, **y_train** ?? hu?n luy?n.  
  
Ta s? l?y c?c th?ng tin n?y nh? sau:  
* X_train: l? **df_train**, nh?ng kh?ng c? tr??ng **relevance**  
* y_train: l? **df_train** ch? v?i tr??ng **relevance**  
* X_test: l? **df_test**, nh?ng kh?ng c? tr??ng **relevance**  
  
Ngo?i ra, ta c?n t?o th?m m?t d? li?u kh?c, l? **id_test**, ?? th?c hi?n vi?c l?y k?t qu? d? do?n v? submit sau n?y.  
  
K?t qu? sau cell n?y l? c?c d? li?u **X_train**, **X_test**, **y_train**, v? **id_test** nh? ?? m? t?.

In [ ]:
# l? df_train, nh?ng kh?ng c? tr??ng relevance 
X_train = df_train.drop('relevance', axis=1)
# l? df_train ch? v?i tr??ng relevance
y_train = df_train['relevance'].values
# l? df_test, nh?ng kh?ng c? tr??ng relevance
X_test = df_test.drop('relevance', axis=1)

# id_test ?? th?c hi?n d? ?o?n v? submit k?t qu? sau n?y
id_test = df_test['id']

# Hu?n luy?n m? h?nh

<a id="model_selection"></a>
## Kh?i t?o m? h?nh 

Gi? ta ch? c?n ??nh ngh?a, th? nghi?m c?c m? h?nh.  
  
??u ti?n, t?i mu?n th? **GradientBoostingRegressor**, m?t m? h?nh ensemble kh? n?i ti?ng, ?? ???c cung c?p trong th? vi?n scikit-learn.  
  
K?t qu? l? m?t object c?a m? h?nh **GBR** n?y.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor()

<a id="find_best_param"></a>
## T?m tham s? t?i ?u

M? h?nh ?? ch?y t?t ???c c?n ph?i c? b? tham s? t?i ?u.  
  
Qua tham kh?o, t?i th?y c? 2 tr??ng th?ng tin quan tr?ng c? th? hi?u ch?nh, ?? l? **n_estimators** v? **max_depth**:
* **n_estimators**: S? l??ng kh?u boost c?n th?c hi?n. V? t?c ??, v? qua tham kh?o, t?i mu?n l?y n_estimators l? c?c s? nguy?n trong kho?ng 30-40, v?i h?m **numpy.linspace**.     
* **max_depth**: ?? s?u gi?i h?n c?a c?c node. Theo tham kh?o, t?i th? nghi?m tr?n c?c s? {2, 4, 6, 8, 10}, v?i h?m **numy.linspace**.     
  
K?t qu? c?a cell n?y l? m?t dictionary mang c?c tham s? c?n hi?u ch?nh cho cell sau.

In [ ]:
param_grid = {
    'n_estimators' : np.array([int(e) for e in np.linspace(30, 40, 11)]),
    'max_depth': np.array([int(e) for e in np.linspace(2, 10, 5)])
}

Gi? ta c?n ch?y m? h?nh nhi?u l?n ?? t?m ???c b? tham s? t?i ?u.  
  
Th? vi?n scikit-learn ?? cung c?p class **GridSearchCV** cho vi?c n?y. Ta s? truy?n v?o ??i t??ng c?n t?i ?u l? **gbr** ?? ??nh ngh?a b?n tr?n, **param_gid=param_grid** ?? ??nh ngh?a, l? b? c?c tham s? c?n t?m t?i ?u, h?m ?i?m **scorring=RMSE** ?? ??nh ngh?a tr??c ??, v?i **cv=3**, l? 3 l?n ki?m ??nh ch?o **cross-validation**.  
  
K?t qu? cell n?y l? ??i t??ng gs ?? t?m b? tham s? t?i ?u.

In [ ]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(estimator=gbr, param_grid=param_grid, cv=3, scoring=RMSE)

Gi? c?ng vi?c c?a ta l? ch?y object **gs** v?i **X_train**, ??i chi?u tr?n **y_train**, ?? t?m ra b? tham s? t?i ?u.  
  
Ta s? d?ng h?m **gs.fit(X_train, y_train)**.  
  
K?t qu? c?a cell n?y l? object **gs** ?? ch?a nh?ng tham s?, v? ?i?m t?i ?u c? ???c trong c?c kh?u cross-validation.  

In [ ]:
%time _ = gs.fit(X_train, y_train)

Nh? ?? n?i ? cell tr?n, gi? ta ch? vi?c in ra b? tham s? t?i ?u.  
  
Ta s? g?i **gs.best_params_** ?? in ra b? tham s? t?i ?u, v? **gs.best_score_** ?? in ra ?i?m t?i ?u ?? ??t ???c.  
  
K?t qu? c?a cell n?y s? ???c l?y ?? hu?n luy?n m? h?nh cu?i v? d? ?o?n k?t qu? cu?i c?ng, n?p l?n kaggle ?? t?nh ?i?m.

In [ ]:
gs.best_params_, gs.best_score_

<a id="best_param_training"></a>
## Hu?n luy?n m? h?nh v?i tham s? t?i ?u

Gi? ta c?n ??nh ngh?a m? h?nh cu?i c?ng, d?a tr?n b? tham s? t?i ?u ?? c?.  
  
Ta kh?i t?o l?i **gbr_best** b?ng **GradientBoostingRegressor**, v?i b? tham s? t?i ?u c? b?n tr?n.  
Sau c?ng, ta g?i h?m **gbr_best.fit(X_train, y_train)** ?? hu?n luy?n m? h?nh.  
  
K?t qu? l? m? h?nh GradientBoostingRegressor t?i ?u m? ta s? s? d?ng ?? d? ?o?n d? li?u test, c?n n?p l?n l?y k?t qu?.

In [ ]:
gbr_best = GradientBoostingRegressor(max_depth=gs.best_params_['max_depth'],
                                     n_estimators=gs.best_params_['n_estimators'])

gbr_best.fit(X_train, y_train)

<a id="prediction"></a>
## D? ?o?n k?t qu? v? n?p

Gi? c?ng vi?c c?a l? c?n d? ?o?n k?t qu? c?a **X_test**.  
  
Ta g?i **gbr_best.predit(X_test)** ?? ???c k?t qu? d? ?o?n.  
  
K?t qu? l? **y_pred** mang c?c gi? tr? **relevance** m? ta d? ?o?n ???c.

In [ ]:
y_pred = gbr_best.predict(X_test)

Cu?i c?ng, ta c?n l?u l?i file output theo ??ng format ?? n?p.  
  
Ta s? d?ng h?m **pandas.DataFrame.to_csv** ?? l?u l?i file, v?i 2 tr??ng th?ng tin l? **id** v? **relevance** - **y_pred** ?? t?nh ???c.  

In [ ]:
pd.DataFrame({"id": id_test, "relevance": y_pred}).to_csv('submission_gbr.csv', index=False)

<a id="result"></a>
# K?t qu?  
  
Private score l? **0.48009**